#Mounting Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
cd /content/drive/MyDrive/33project/05.AR/06.FAN/FAN_pytorch

/content/drive/MyDrive/33project/05.AR/06.FAN/FAN_pytorch


#Installing Depedencies

In [9]:
!pip3 install --upgrade --no-cache-dir gdown
# !pip3 install torchvision==0.7.0  # 현재 Colab과 호환되지 않으므로 제외
!pip3 install progress
!pip3 install torchfile

#Defining Variables

In [10]:
dataset='data/300W_LP'
# model_path='pretrain'
model_file='3D_model_best.pth.tar'
detectmodelfile='s3fd_convert.pth'
input='test-images'
data='data/300W_LP'

snapshot=1
epochs=40
workers=4
train_batch=40
val_batch=40
checkpoint='checkpoint'
resume=False
lr=1e-4
momentum=0.0
weight_decay=0.0

netType='FAN'
pointType='2D'
nStack=4
nModules=1
nHgDepth=4
nFeats=256
nclasses=68
schedule=[15, 30]
gamma=0.1
retrain=''
optimState=''

scale_factor=0.3
rot_factor=30
evaluation=False
debug=False
flip=False
start_epoch=0

#Importing Libraries

In [11]:
from __future__ import print_function
import os
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
import math
from enum import Enum
from skimage import io
from skimage import color
import numpy as np
import cv2
try:
    import urllib.request as request_file
except BaseException:
    import urllib as request_file

import os
import sys
import time
import matplotlib
matplotlib.use('Agg')
from progress.bar import Bar
import matplotlib.pyplot as plt

import torch
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.optim
import torchvision.datasets as datasets

import models
from utils import *
from datasets import W300LP, VW300, AFLW2000, LS3DW
from utils.logger import Logger, savefig
from utils.imutils import batch_with_heatmap
from utils.evaluation import accuracy, AverageMeter, final_preds, calc_metrics, calc_dists
from utils.misc import adjust_learning_rate, save_checkpoint, save_pred


#Downloading Dataset

In [12]:
if os.path.exists(dataset) is False:
  print("Downloading Dataset")

  !gdown https://drive.google.com/u/0/uc?id=0B7OEHD3T4eCkVGs0TkhUWFN6N1k&export=download
  !wget https://www.adrianbulat.com/downloads/FaceAlignment/landmarks.zip
  !unzip 300W-LP.zip -d data/
  !unzip landmarks.zip -d data/300W_LP/

else:
  print('Dataset already Downloaded')

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: data/300W_LP/HELEN_Flip/HELEN_2365877276_1_1.mat  
  inflating: data/300W_LP/HELEN_Flip/HELEN_2365877276_1_10.jpg  
  inflating: data/300W_LP/HELEN_Flip/HELEN_2365877276_1_10.mat  
  inflating: data/300W_LP/HELEN_Flip/HELEN_2365877276_1_11.jpg  
  inflating: data/300W_LP/HELEN_Flip/HELEN_2365877276_1_11.mat  
  inflating: data/300W_LP/HELEN_Flip/HELEN_2365877276_1_12.jpg  
  inflating: data/300W_LP/HELEN_Flip/HELEN_2365877276_1_12.mat  
  inflating: data/300W_LP/HELEN_Flip/HELEN_2365877276_1_2.jpg  
  inflating: data/300W_LP/HELEN_Flip/HELEN_2365877276_1_2.mat  
  inflating: data/300W_LP/HELEN_Flip/HELEN_2365877276_1_3.jpg  
  inflating: data/300W_LP/HELEN_Flip/HELEN_2365877276_1_3.mat  
  inflating: data/300W_LP/HELEN_Flip/HELEN_2365877276_1_4.jpg  
  inflating: data/300W_LP/HELEN_Flip/HELEN_2365877276_1_4.mat  
  inflating: data/300W_LP/HELEN_Flip/HELEN_2365877276_1_5.jpg  
  inflating: data/300W_LP/HELEN_Flip/HELEN_2365877276_1_5.mat 

In [13]:
import os
print(os.getcwd())
print(os.path.abspath("data/300W_LP"))

/content/drive/MyDrive/33project/05.AR/06.FAN/FAN_pytorch
/content/drive/MyDrive/33project/05.AR/06.FAN/FAN_pytorch/data/300W_LP


#Downloading PreTrained Models

In [14]:
if os.path.exists(model_file) is False:
  print("Downloading PreTrained Models")
  !gdown https://drive.google.com/u/0/uc?id=1RPqjxmli69V9vpez3gRFo-6jdTppX5gm&export=download
  !wget https://www.adrianbulat.com/downloads/python-fan/s3fd_convert.pth
else:
  print('Models already Downloaded')

Using cookies from /root/.cache/gdown/cookies.txt
Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1RPqjxmli69V9vpez3gRFo-6jdTppX5gm

but Gdown can't. Please check connections and permissions.
--2026-09-29 08:43:46--  https://www.adrianbulat.com/downloads/python-fan/s3fd_convert.pth
Resolving www.adrianbulat.com (www.adrianbulat.com)... 45.136.29.207
Connecting to www.adrianbulat.com (www.adrianbulat.com)|45.136.29.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 89843225 (86M) [application/octet-stream]
Saving to: ‘s3fd_convert.pth’

s3fd_convert.pth    100%[===================>]  85.68M  16.3MB/s    in 6.0s    

2026-09-29 08:43:53 (14.2 MB/s) - ‘s3fd_convert.pth’ saved [8984

#Defining Model

In [15]:
__all__ = ['FAN','ResNetDepth']

def conv3x3(in_planes, out_planes, strd=1, padding=1, bias=False):
    "3x3 convolution with padding"
    return nn.Conv2d(in_planes, out_planes, kernel_size=3,
                     stride=strd, padding=padding, bias=bias)


class ConvBlock(nn.Module):
    def __init__(self, in_planes, out_planes):
        super(ConvBlock, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = conv3x3(in_planes, int(out_planes / 2))
        self.bn2 = nn.BatchNorm2d(int(out_planes / 2))
        self.conv2 = conv3x3(int(out_planes / 2), int(out_planes / 4))
        self.bn3 = nn.BatchNorm2d(int(out_planes / 4))
        self.conv3 = conv3x3(int(out_planes / 4), int(out_planes / 4))

        if in_planes != out_planes:
            self.downsample = nn.Sequential(
                nn.BatchNorm2d(in_planes),
                nn.ReLU(True),
                nn.Conv2d(in_planes, out_planes,
                          kernel_size=1, stride=1, bias=False),
            )
        else:
            self.downsample = None

    def forward(self, x):
        residual = x

        out1 = self.bn1(x)
        out1 = F.relu(out1, True)
        out1 = self.conv1(out1)

        out2 = self.bn2(out1)
        out2 = F.relu(out2, True)
        out2 = self.conv2(out2)

        out3 = self.bn3(out2)
        out3 = F.relu(out3, True)
        out3 = self.conv3(out3)

        out3 = torch.cat((out1, out2, out3), 1)

        if self.downsample is not None:
            residual = self.downsample(residual)

        out3 += residual

        return out3


class Bottleneck(nn.Module):

    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, planes * 4, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out


class HourGlass(nn.Module):
    def __init__(self, nModules, depth, num_features):
        super(HourGlass, self).__init__()
        self.nModules = nModules
        self.depth = depth
        self.features = num_features

        self._generate_network(self.nModules, self.depth)

    def _generate_network(self, nModules, level):
        # Upper branch
        for conv_module in range(nModules):
            self.add_module('b1_' + str(level) + '_' + str(conv_module), ConvBlock(self.features, self.features))

        # Lower branch
        for conv_module in range(nModules):
            self.add_module('b2_' + str(level) + '_' + str(conv_module), ConvBlock(self.features, self.features))

        # HG body recursion
        if level > 1:
            self._generate_network(nModules, level - 1)
        else:
            for conv_module in range(nModules):
                self.add_module('b2_plus_' + str(level) + '_' + str(conv_module), ConvBlock(self.features, self.features))

        for conv_module in range(nModules):
            self.add_module('b3_' + str(level) + '_' + str(conv_module), ConvBlock(self.features, self.features))

    def _forward(self, nModules, level, inp):
        # Upper branch
        up1 = inp
        for conv_module in range(nModules):
            up1 = self._modules['b1_' + str(level) + '_' + str(conv_module)](up1)

        # Lower branch
        low1 = F.max_pool2d(inp, 2, stride=2)
        for conv_module in range(nModules):
            low1 = self._modules['b2_' + str(level) + '_' + str(conv_module)](low1)

        # HG body recursion
        if level > 1:
            low2 = self._forward(nModules, level - 1, low1)
        else:
            low2 = low1
            for conv_module in range(nModules):
                low2 = self._modules['b2_plus_' + str(level) + '_' + str(conv_module)](low2)

        low3 = low2
        for conv_module in range(nModules):
            low3 = self._modules['b3_' + str(level) + '_' + str(conv_module)](low3)

        up2 = F.interpolate(low3, scale_factor=2, mode='nearest')

        # Bring two branches together
        return up1 + up2

    def forward(self, x):
        return self._forward(self.nModules, self.depth, x)


class FAN(nn.Module):

    def __init__(self, nStack=4, nModules=1, nHgDepth=4, num_feats=256, num_classes=68):
        super(FAN, self).__init__()

        self.inplanes = 64
        self.expansion = 2
        self.nStack = nStack        #num of how many HourGlassNet stack to the whole FAN Net
        self.nModules = nModules    #num of ConvBlock==Residual before HG block
        self.nHgDepth = nHgDepth    #num of recursion in HourGlassNet
        self.num_feats = num_feats
        self.num_classes = num_classes

        # Base part
        self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(self.inplanes)
        self.conv2 = ConvBlock(self.inplanes, self.inplanes*self.expansion)
        self.conv3 = ConvBlock(self.inplanes*self.expansion, self.inplanes*self.expansion)
        self.conv4 = ConvBlock(self.inplanes*self.expansion, self.num_feats)

        # Stacking part
        for hg_stack in range(self.nStack):
            self.add_module('m' + str(hg_stack), HourGlass(self.nModules, self.nHgDepth, self.num_feats))

            for conv_module in range(self.nModules):
                self.add_module('top_m_' + str(hg_stack) + '_' + str(conv_module), ConvBlock(self.num_feats, self.num_feats))

            self.add_module('conv_last' + str(hg_stack),
                            nn.Conv2d(self.num_feats, self.num_feats, kernel_size=1, stride=1, padding=0))
            self.add_module('bn_end' + str(hg_stack), nn.BatchNorm2d(self.num_feats))
            self.add_module('l' + str(hg_stack), nn.Conv2d(self.num_feats,
                                                            self.num_classes, kernel_size=1, stride=1, padding=0))

            if hg_stack < self.nStack - 1:
                self.add_module(
                    'bl' + str(hg_stack), nn.Conv2d(self.num_feats, self.num_feats, kernel_size=1, stride=1, padding=0))
                self.add_module('al' + str(hg_stack), nn.Conv2d(self.num_classes,
                                                                 self.num_feats, kernel_size=1, stride=1, padding=0))

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)), True)
        x = F.max_pool2d(self.conv2(x), 2, stride=2)
        x = self.conv3(x)
        x = self.conv4(x)

        previous = x

        outputs = []
        for i in range(self.nStack):
            hg = self._modules['m' + str(i)](previous)

            # Residual layers at output resolution
            ll = hg
            for conv_module in range(self.nModules):
                ll = self._modules['top_m_' + str(i) + '_' + str(conv_module)](ll)

            # Linear layer to produce first set of predictions
            ll = F.relu(self._modules['bn_end' + str(i)]
                        (self._modules['conv_last' + str(i)](ll)), True)

            # Predict heatmaps
            tmp_out = self._modules['l' + str(i)](ll)
            outputs.append(tmp_out)

            if i < self.nStack - 1:
                ll = self._modules['bl' + str(i)](ll)
                tmp_out_ = self._modules['al' + str(i)](tmp_out)
                previous = previous + ll + tmp_out_

        return outputs


class ResNetDepth(nn.Module):

    def __init__(self, block=Bottleneck, layers=[3, 8, 36, 3], num_classes=68):
        self.inplanes = 64
        super(ResNetDepth, self).__init__()
        self.conv1 = nn.Conv2d(3 + 68, 64, kernel_size=7, stride=2, padding=3,
                               bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.avgpool = nn.AvgPool2d(7)
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x

class FaceAlignment:
    def __init__(self,
                modelfilename="checkpoint_100.pth.tar",
                nStack=4,
                device='cuda',
                flip_input=False,
                face_detector='sfd',
                facedetectmodelfile="s3fd_convert.pth",
                verbose=False):

        self.device = device
        self.flip_input = flip_input
        self.verbose = verbose
        import os

        if not os.path.isfile(facedetectmodelfile):
            print("The face detection CNN model [%s] not exists." % facedetectmodelfile)
            return None

        if 'cuda' in device:
            torch.backends.cudnn.benchmark = True

        # Get the face detector
        face_detector_module = __import__('detection.' + face_detector,
                                          globals(), locals(), [face_detector], 0)
        self.face_detector = face_detector_module.FaceDetector(device=device, path_to_detector=facedetectmodelfile, verbose=verbose)

        # Initialise the face alignemnt networks
        self.face_alignment_net = nn.DataParallel(FAN(nStack))
        model_path = modelfilename

        if not os.path.isfile(model_path):
            print("model:%s not exists." % model_path)
            return None
        fan_weights = torch.load(
            model_path,
            map_location=lambda storage,
            loc: storage)


        self.face_alignment_net.load_state_dict(fan_weights['state_dict'])
        self.face_alignment_net = self.face_alignment_net.module

        self.face_alignment_net.to(device)
        self.face_alignment_net.eval()

    def get_landmarks(self, image_or_path, detected_faces=None):

        return self.get_landmarks_from_image(image_or_path, detected_faces)

    def get_landmarks_from_image(self, image_or_path, detected_faces=None):

        if isinstance(image_or_path, str):
            try:
                image = io.imread(image_or_path)
            except IOError:
                print("error opening file :: ", image_or_path)
                return None
        else:
            image = image_or_path

        if image.ndim == 2:
            image = color.gray2rgb(image)
        elif image.ndim == 4:
            image = image[..., :3]

        reference_scale = 200
        if detected_faces is None:
            detected_faces = self.face_detector.detect_from_image(image[..., ::-1].copy())
            reference_scale = self.face_detector.reference_scale

        if len(detected_faces) == 0:
            print("Warning: No faces were detected.")
            return None

        torch.set_grad_enabled(False)
        landmarks = []
        landmarks_in_crops = []
        img_crops = []
        # print('detected_faces num:{}'.format(len(detected_faces)))
        # print('detected_faces:{}'.format(detected_faces))

        image = im_to_torch(image)

        for i, d in enumerate(detected_faces):
            center = torch.FloatTensor(
                [d[2] - (d[2] - d[0]) / 2.0, d[3] -
                 (d[3] - d[1]) / 2.0])
            center[1] = center[1] + (d[3] - d[1]) * 0.12
            hw = max(d[2] - d[0], d[3] - d[1])
            scale_x = float(hw / reference_scale)
            scale_y = float(hw / reference_scale)

            #try to verify this part
            inp = crop(image, center, [scale_x, scale_y], reference_scale)

            io.imsave('crop_%s.jpg' % i,im_to_numpy(inp))
            img_crops.append(im_to_numpy(inp))

            inp = inp.to(self.device)
            inp.unsqueeze_(0)

            out = self.face_alignment_net(inp)[-1].detach()
            if self.flip_input:
                out += flip(self.face_alignment_net(flip(inp))
                            [-1].detach(), is_label=True)
            out = out.cpu()

            pts, pts_img = get_preds_fromhm(out, [center], [[scale_x, scale_y]], [reference_scale])
            pts, pts_img = pts.view(68, 2) * 4, pts_img.view(68, 2)

            landmarks.append(pts_img.numpy())
            landmarks_in_crops.append(pts.numpy())

        return landmarks, detected_faces, landmarks_in_crops, img_crops

    def get_landmarks_from_directory(self, path, extensions=['.jpg', '.png'], recursive=True, show_progress_bar=True):
        detected_faces = self.face_detector.detect_from_directory(path, extensions, recursive, show_progress_bar)

        predictions = {}
        for image_path, bounding_boxes in detected_faces.items():
            image = io.imread(image_path)
            preds, detected_faces = self.get_landmarks_from_image(image, bounding_boxes)
            predictions[image_path] = preds

        return predictions

#Training Script

In [16]:
model_names = sorted(
    name for name in models.__dict__
    if not name.startswith("__") and callable(models.__dict__[name]))
# torch.setdefaulttensortype('torch.FloatTensor')

best_acc = 0.
best_auc = 0.
idx = range(1, 69, 1)


def get_loader(data):
    return {
        '300W_LP': W300LP,
        'LS3D-W/300VW-3D': VW300,
        'AFLW2000': AFLW2000,
        'LS3D-W': LS3DW,
    }[data[5:]]


def trainFAN(checkpoint,lr,start_epoch):
    global best_acc
    global best_auc

    if not os.path.exists(checkpoint):
        os.makedirs(checkpoint)

    print("==> Creating model '{}-{}', stacks={} modules={}, hgdepths={} feats={} classes={}".format(
        netType, pointType, nStack, nModules, nHgDepth, nFeats, nclasses))

    print("=> Models will be saved at: {}".format(checkpoint))

    # nStack=4, nModules=1, nHgDepth=4, num_feats=256, num_classes=68
    model = models.__dict__[netType](
        nStack=nStack,
        nModules=nModules,
        nHgDepth=nHgDepth,
        num_feats=nFeats,
        num_classes=nclasses)

    model = torch.nn.DataParallel(model).cuda()

    criterion = torch.nn.MSELoss(size_average=True).cuda()

    optimizer = torch.optim.RMSprop(
        model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

    title = checkpoint.split('/')[-1] + ' on ' + data.split('/')[-1]

    Loader = get_loader(data)

    val_loader = torch.utils.data.DataLoader(
        Loader('A'),
        batch_size=val_batch,
        shuffle=False,
        num_workers=workers,
        pin_memory=True)

    if resume:
        if os.path.isfile(resume):
            print("=> Loading checkpoint '{}'".format(resume))
            checkpoint = torch.load(resume)
            start_epoch = checkpoint['epoch']
            best_acc = checkpoint['best_acc']
            model.load_state_dict(checkpoint['state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer'])
            print("=> Loaded checkpoint '{}' (epoch {})".format(resume, checkpoint['epoch']))
            logger = Logger(os.path.join(checkpoint, 'log.txt'), title=title, resume=True)
        else:
            print("=> no checkpoint found at '{}'".format(resume))
            logger = Logger(os.path.join(checkpoint, 'log.txt'), title=title)
            logger.set_names(['Epoch', 'LR', 'Train Loss', 'Valid Loss', 'Train Acc', 'Val Acc', 'AUC'])
    else:
        logger = Logger(os.path.join(checkpoint, 'log.txt'), title=title)
        logger.set_names(['Epoch', 'LR', 'Train Loss', 'Valid Loss', 'Train Acc', 'Val Acc', 'AUC'])

    cudnn.benchmark = True
    print('=> Total params: %.2fM' % (sum(p.numel() for p in model.parameters()) / (1024. * 1024)))

    train_loader = torch.utils.data.DataLoader(
        Loader('train'),
        batch_size=train_batch,
        shuffle=True,
        num_workers=workers,
        pin_memory=True)
    lr = lr
    for epoch in range(start_epoch, epochs):
        lr = adjust_learning_rate(optimizer, epoch, lr, schedule, gamma)
        print('=> Epoch: %d | LR %.8f' % (epoch + 1, lr))
        sys.stdout.flush()

        train_loss, train_acc = train(train_loader, model, criterion, optimizer, netType,
                                      debug, flip)
        # do not save predictions in model file
        valid_loss, valid_acc, predictions, valid_auc = validate(val_loader, model, criterion, netType,
                                                      debug, flip)

        logger.append([int(epoch + 1), lr, train_loss, valid_loss, train_acc, valid_acc, valid_auc])

        is_best = valid_auc >= best_auc
        best_auc = max(valid_auc, best_auc)
        save_checkpoint(
            {
                'epoch': epoch + 1,
                'netType': netType,
                'state_dict': model.state_dict(),
                'best_acc': best_auc,
                'optimizer': optimizer.state_dict(),
            },
            is_best,
            predictions,
            checkpoint=checkpoint,
            filename='checkpoint.pth.tar',
            snapshot=snapshot)

    logger.close()
    logger.plot(['AUC'])
    savefig(os.path.join(checkpoint, 'log.eps'))


def train(loader, model, criterion, optimizer, netType, debug=False, flip=False):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    acces = AverageMeter()

    model.train()
    end = time.time()

    gt_win, pred_win = None, None
    bar = Bar('Training', max=len(loader))
    for i, (inputs, target) in enumerate(loader):
        data_time.update(time.time() - end)

        input_var = torch.autograd.Variable(inputs.cuda())
        target_var = torch.autograd.Variable(target.cuda(non_blocking=True))

        if debug:
            gt_batch_img = batch_with_heatmap(inputs, target)
            # pred_batch_img = batch_with_heatmap(inputs, score_map)
            if not gt_win or not pred_win:
                plt.subplot(121)
                gt_win = plt.imshow(gt_batch_img)

            else:
                gt_win.set_data(gt_batch_img)
                # pred_win.set_data(pred_batch_img)
            plt.pause(.05)
            plt.draw()

        output = model(input_var)
        score_map = output[-1].data.cpu()


    return losses.avg, acces.avg


def validate(loader, model, criterion, netType, debug, flip):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    acces = AverageMeter()
    end = time.time()

    # predictions
    predictions = torch.Tensor(loader.dataset.__len__(), 68, 2)

    model.eval()
    gt_win, pred_win = None, None
    bar = Bar('Validating', max=len(loader))
    all_dists = torch.zeros((68, loader.dataset.__len__()))
    for i, (inputs, target, meta) in enumerate(loader):
        data_time.update(time.time() - end)

        input_var = torch.autograd.Variable(inputs.cuda())
        target_var = torch.autograd.Variable(target.cuda(non_blocking=True))

        output = model(input_var)
        score_map = output[-1].data.cpu()


        # intermediate supervision
        loss = 0
        for o in output:
            loss += criterion(o, target_var)
        acc, batch_dists = accuracy(score_map, target.cpu(), idx, thr=0.07)
        all_dists[:, i * val_batch:(i + 1) * val_batch] = batch_dists

        #preds = final_preds(score_map, meta['center'], meta['scale'], meta['reference_scale'], [64, 64])
        pts, pts_img = get_preds_fromhm(score_map, meta['center'], meta['scale'], meta['reference_scale'])
        preds = pts_img
        for n in range(score_map.size(0)):
            predictions[meta['index'][n], :, :] = preds[n, :, :]

        if debug:
            gt_batch_img = batch_with_heatmap(inputs, target)
            pred_batch_img = batch_with_heatmap(inputs, score_map)
            if not gt_win or not pred_win:
                plt.subplot(121)
                gt_win = plt.imshow(gt_batch_img)
                plt.subplot(122)
                pred_win = plt.imshow(pred_batch_img)
            else:
                gt_win.set_data(gt_batch_img)
                pred_win.set_data(pred_batch_img)
            plt.pause(.05)
            plt.draw()

        losses.update(loss.data, inputs.size(0))# replaced lose.data[] by lose.data
        acces.update(acc[0], inputs.size(0))

        batch_time.update(time.time() - end)
        end = time.time()

        bar.suffix = '({batch}/{size}) Data: {data:.6f}s | Batch: {bt:.3f}s | Total: {total:} | ETA: {eta:} | Loss: {loss:.4f} | Acc: {acc: .4f}'.format(
            batch=i + 1,
            size=len(loader),
            data=data_time.val,
            bt=batch_time.val,
            total=bar.elapsed_td,
            eta=bar.eta_td,
            loss=losses.avg,
            acc=acces.avg)
        bar.next()

    bar.finish()
    mean_error = torch.mean(all_dists)
    auc = calc_metrics(all_dists) # this is auc of predicted maps and target.
    print("=> Mean Error: {:.2f}, AUC@0.07: {} based on maps".format(mean_error*100., auc))
    sys.stdout.flush()
    return losses.avg, acces.avg, predictions, auc



trainFAN(checkpoint,lr,start_epoch)


==> Creating model 'FAN-2D', stacks=4 modules=1, hgdepths=4 feats=256 classes=68
=> Models will be saved at: checkpoint
=> loaded validation set, 0 images were found
	computing mean and std for the first time, it may takes a while, drink a cup of coffe...
=> Total params: 22.72M
=> loaded train set, 206 images were found
	Mean: nan, nan, nan
	Std:  nan, nan, nan
=> Epoch: 1 | LR 0.00010000


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/loss.py:44: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  self.reduction: str = _Reduction.legacy_get_string(size_average, reduce)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be 

ValueError: Caught ValueError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/content/drive/MyDrive/33project/05.AR/06.FAN/FAN_pytorch/datasets/W300LP.py", line 60, in __getitem__
    inp, out, pts, c, s, reference_scale = self.generateSampleFace(index)
                                           ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^
  File "/content/drive/MyDrive/33project/05.AR/06.FAN/FAN_pytorch/datasets/W300LP.py", line 70, in generateSampleFace
    main_pts = torchfile.load(
        os.path.join(self.img_folder, 'landmarks', self.anno[idx].split('_')[0],
                     self.anno[

#Inference Script
###NOTE : If cuda runs out of memory , restart runtime and run again.
### After Training , Cuda may run out of memory for Inference.

In [17]:
fa = FaceAlignment(modelfilename=model_file, facedetectmodelfile=detectmodelfile)
if fa:
    for path in os.listdir(input):
      img_in = io.imread(input+'/'+path)
      img = img_in
      preds, detected_faces, preds_in_crops, img_crops = fa.get_landmarks_from_image(img)
      for k,d in enumerate(detected_faces):
          cv2.rectangle(img_in,(int(d[0]),int(d[1])),(int(d[2]),int(d[3])),(255,255,255))
          landmark = preds[k]
          for i in range(landmark.shape[0]):
              pts = landmark[i]
              cv2.circle(img_in, (int(pts[0]), int(pts[1])),5,(0,255,0), -1, 8)
              cv2.putText(img_in,str(i),(int(pts[0]),int(pts[1])),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,2555,255))
      # io.imsave(f'prediction_{path}',img_in)
      img=Image.open(f'prediction_{path}')
      display(img)



else:
    print("FaceAlignment init error!")

model:3D_model_best.pth.tar not exists.


TypeError: Cannot handle this data type: (1, 1, 3), <f4